In [1]:
import random

import cv2
import face_recognition
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets

In [2]:
random.seed(1000)
torch.manual_seed(1000)

In [3]:
# Define transforms (resize and convert to tensor)
transform = transforms.Compose([
    # transforms.Resize((128, 128)),  # Resize all images to 128x128
    transforms.ToTensor(),  # Convert images to PyTorch tensors
])

In [4]:
celeba_dataset = datasets.CelebA('data', target_type='identity', split='all', transform=transform, download=False)

In [5]:
# Get identity information
identity_data = celeba_dataset.identity
identity_data

tensor([[ 2880],
        [ 2937],
        [ 8692],
        ...,
        [ 9852],
        [ 5570],
        [10101]])

In [6]:
# Create a mapping from identity to all their image indices
identity_to_images = {}

for idx, person_id in enumerate(identity_data):
    identity = person_id.item()
    if identity not in identity_to_images:
        identity_to_images[identity] = []
    identity_to_images[identity].append(idx)

identity_to_images

{2880: [0,
  403,
  3414,
  4389,
  18061,
  25243,
  27770,
  39392,
  47977,
  49141,
  52384,
  52622,
  53183,
  53310,
  55833,
  58187,
  61430,
  68153,
  84704,
  90936,
  96323,
  100989,
  103727,
  108340,
  110375,
  122438,
  131730,
  134006,
  139105,
  140934],
 2937: [1,
  11436,
  16334,
  17120,
  24290,
  37081,
  45317,
  46843,
  48359,
  55890,
  57356,
  58208,
  58399,
  59637,
  60924,
  63241,
  63615,
  66808,
  77345,
  95866,
  99876,
  105286,
  108660,
  114335,
  114624,
  117709,
  120279,
  125139,
  142600,
  152379],
 8692: [2,
  15647,
  33839,
  38886,
  49971,
  52373,
  53296,
  61068,
  69457,
  70479,
  73664,
  74690,
  74907,
  79722,
  84417,
  86207,
  87939,
  92114,
  101758,
  102190,
  106610,
  133004,
  134208,
  134271,
  134679,
  138522,
  140746,
  148650,
  153309,
  154305],
 5805: [3,
  1777,
  10190,
  13675,
  27056,
  27558,
  29582,
  30444,
  32972,
  41165,
  49164,
  57459,
  68649,
  71571,
  75316,
  79294,
  82761,
 

In [7]:
# Select only 100 identities, with fixed random seed
num_identities = 100

sampled_identities = random.sample(list(identity_to_images.keys()), 
                                   num_identities * 2)
sampled_identities

[6732,
 5266,
 7208,
 631,
 995,
 1363,
 4529,
 540,
 6211,
 7756,
 45,
 136,
 9156,
 4903,
 4416,
 3383,
 269,
 9278,
 6733,
 517,
 1495,
 3870,
 8502,
 5249,
 659,
 4284,
 2226,
 9982,
 8487,
 3358,
 10021,
 7,
 3771,
 5401,
 7821,
 5484,
 2464,
 10170,
 9955,
 5293,
 389,
 2476,
 1291,
 8852,
 8430,
 4866,
 4838,
 6634,
 7588,
 8970,
 9957,
 6940,
 8179,
 1761,
 8167,
 8824,
 8628,
 183,
 5031,
 9508,
 3396,
 6835,
 6381,
 693,
 4094,
 1956,
 329,
 7361,
 3336,
 4301,
 1035,
 6186,
 6100,
 3327,
 9366,
 2485,
 4165,
 399,
 1051,
 6673,
 5348,
 3139,
 2521,
 9393,
 2698,
 1261,
 2542,
 5360,
 3065,
 8994,
 5794,
 8533,
 5946,
 4251,
 503,
 8007,
 4112,
 5309,
 3658,
 4476,
 7047,
 3768,
 9672,
 9353,
 4316,
 7126,
 6631,
 7277,
 8642,
 8478,
 6821,
 9542,
 4483,
 2335,
 828,
 5607,
 7512,
 1205,
 3890,
 1852,
 4625,
 7124,
 3085,
 2445,
 2170,
 5273,
 4778,
 7063,
 2901,
 4626,
 6553,
 1489,
 7797,
 9049,
 6961,
 5490,
 611,
 8339,
 4863,
 9096,
 971,
 2467,
 7969,
 5617,
 7472,
 749

In [8]:
sampled_known_identities = sampled_identities[:num_identities]
sampled_unknown_identities = sampled_identities[num_identities:]

In [9]:
def tensor_to_opencv(img):
    # Step 1: Move to CPU if it's on GPU and convert to NumPy
    numpy_image = img.cpu().numpy()

    # Step 2: Transpose from (C, H, W) to (H, W, C) to match OpenCV format
    numpy_image = np.transpose(numpy_image, (1, 2, 0))

    # Step 3: Convert from [0, 1] to [0, 255] range if necessary and convert to uint8
    numpy_image = (numpy_image * 255).astype(np.uint8)

    # Step 4: Convert to OpenCV BGR format if needed (PyTorch uses RGB by default)
    return cv2.cvtColor(numpy_image, cv2.COLOR_RGB2BGR)


In [10]:
# Create the train and test datasets
train_indices = list()
test_indices = list()

# Dictionary to store encodings for each identity
face_encodings = {}

for face_id in sampled_known_identities:
    image_indices = identity_to_images[face_id]
    if len(image_indices) <= 1:
        continue

    # Select an image with detected face (no detection -> no encoding)
    result = []
    while len(result) != 1:
        # Randomly select one image for training (encoding)
        train_index = random.choice(image_indices)
        image, identity = celeba_dataset[train_index]

        # Get face encodings
        image = tensor_to_opencv(image)
        result = face_recognition.face_encodings(image)

    # Store index and encodings
    train_indices.append(train_index)
    face_encodings[identity.item()] = result[0]

    # The rest of the images go into the test set
    test_indices.extend(idx for idx in image_indices if idx != train_index)

In [11]:
len(test_indices)

1775

In [12]:
# Create test datasets for unknown faces
unknown_test_indices = list()

for face_id in sampled_unknown_identities:
    unknown_test_indices.extend(identity_to_images[face_id])

In [13]:
len(unknown_test_indices)

1986

In [14]:
# Create PyTorch subsets for the train and test datasets
test_subset = Subset(celeba_dataset, test_indices)

# Create DataLoader for the test set
test_loader = DataLoader(test_subset, batch_size=16, shuffle=True)

In [15]:
# Construct known encodings mapping (map[index] -> identity)
known_face_encodings_map = []
known_face_encodings = []
for k, v in face_encodings.items():
    known_face_encodings_map.append(k)
    known_face_encodings.append(v)

In [16]:
corrects = 0
total = 0
tolerance = 0.5

# Test phase (compare test images with known identities)
for idx in test_indices:
    image, identity = celeba_dataset[idx]
    total += 1
    
    # Get face encodings
    image = tensor_to_opencv(image)
    result = face_recognition.face_encodings(image)
    
    # No detected face, failed
    if len(result) != 1:
        continue

    # Face matching and distance calculation
    detected_face_encodings = result[0]
    matches = face_recognition.compare_faces(
        known_face_encodings=known_face_encodings,
        face_encoding_to_check=detected_face_encodings,
        tolerance=tolerance
    )
    distances = face_recognition.face_distance(
        face_encodings=known_face_encodings,
        face_to_compare=detected_face_encodings
    )
    
    # Closest matching face
    best_match_index = np.argmin(distances)
    if matches[best_match_index]:
        
        # Matches correct identity
        if known_face_encodings_map[best_match_index] == identity.item():
            corrects += 1

# Test phase (compare test images with unknown identities)
for idx in unknown_test_indices:
    image, _ = celeba_dataset[idx]
    total += 1
    
    # Get face encodings
    image = tensor_to_opencv(image)
    result = face_recognition.face_encodings(image)
    
    # No detected face, failed
    if len(result) != 1:
        continue

    # Face matching and distance calculation
    detected_face_encodings = result[0]
    matches = face_recognition.compare_faces(
        known_face_encodings=known_face_encodings,
        face_encoding_to_check=detected_face_encodings,
        tolerance=tolerance
    )
    distances = face_recognition.face_distance(
        face_encodings=known_face_encodings,
        face_to_compare=detected_face_encodings
    )
    
    # Closest matching face
    best_match_index = np.argmin(distances)
    if not matches[best_match_index]:
        corrects += 1

In [17]:
print(f'{tolerance = }')
print(f'{corrects = }')
print(f'{total = }')
accuracy = corrects / total
print(f'{accuracy = :2%}')

tolerance = 0.5
corrects = 2911
total = 3761
accuracy = 77.399628%
